**Model Validation - Assignment 2**

In [11]:
# Cell 1: imports and raw data
import numpy as np
import pandas as pd
from scipy.stats import norm

prices = pd.read_csv('project9/raw_price_data-1.csv', index_col=0, parse_dates=True)
rates  = pd.read_csv('project9/DTB3-1.csv', index_col=0, parse_dates=True)

equity_cols = ['INGA.AS', 'SHEL', 'XOM', '^AEX', '^GSPC']
rates.columns = ['DTB3']  # make sure the column name is correct

print("Raw price rows:", len(prices))
print("Raw rate rows:", len(rates))
print("\nRaw missing counts:")
print(prices.isna().sum())
print(rates.isna().sum())

Raw price rows: 2086
Raw rate rows: 2087

Raw missing counts:
EURUSD=X    24
INGA.AS     36
SHEL        74
XOM         74
^AEX        41
^GSPC       74
dtype: int64
DTB3    84
dtype: int64


In [12]:
# Cell 2: cleaning
prices_clean = prices.copy()
rates_clean = rates.copy()

for col in equity_cols:
    prices_clean[col] = prices_clean[col].ffill(limit=1)

prices_clean['EURUSD=X'] = prices_clean['EURUSD=X'].ffill(limit=5)
rates_clean['DTB3'] = rates_clean['DTB3'].ffill(limit=1)

merged = prices_clean.join(rates_clean, how='left')

print("\nRows before dropna:", len(merged))
print("Rows with any NaN:", merged.isna().any(axis=1).sum())
print("Dates dropped:")
print(merged[merged.isna().any(axis=1)].index.tolist())

cleaned_prices = merged.dropna().copy()

print("\nCleaned price rows:", len(cleaned_prices))
print("Date range:", cleaned_prices.index.min().date(), "to", cleaned_prices.index.max().date())


Rows before dropna: 2086
Rows with any NaN: 22
Dates dropped:
[Timestamp('2006-01-02 00:00:00'), Timestamp('2006-04-17 00:00:00'), Timestamp('2006-12-26 00:00:00'), Timestamp('2007-01-02 00:00:00'), Timestamp('2007-04-09 00:00:00'), Timestamp('2007-12-26 00:00:00'), Timestamp('2008-03-24 00:00:00'), Timestamp('2008-08-18 00:00:00'), Timestamp('2008-08-19 00:00:00'), Timestamp('2008-08-20 00:00:00'), Timestamp('2008-08-21 00:00:00'), Timestamp('2008-08-22 00:00:00'), Timestamp('2008-08-25 00:00:00'), Timestamp('2008-12-26 00:00:00'), Timestamp('2009-04-13 00:00:00'), Timestamp('2010-04-05 00:00:00'), Timestamp('2011-04-25 00:00:00'), Timestamp('2012-04-09 00:00:00'), Timestamp('2012-10-30 00:00:00'), Timestamp('2012-12-26 00:00:00'), Timestamp('2013-04-01 00:00:00'), Timestamp('2013-12-26 00:00:00')]

Cleaned price rows: 2064
Date range: 2006-01-03 to 2013-12-30


In [13]:
# Cell 3: returns
CREDIT_SPREAD = 0.01

log_returns = pd.DataFrame(index=cleaned_prices.index)

for col in equity_cols + ['EURUSD=X']:
    log_returns[col] = np.log(cleaned_prices[col] / cleaned_prices[col].shift(1))

log_returns['log_loan'] = np.log(
    1 + (cleaned_prices['DTB3'] / 100 + CREDIT_SPREAD) / 252
)

log_returns = log_returns.dropna().copy()

print("\nLog return rows:", len(log_returns))
print(log_returns.describe().round(6))


Log return rows: 2063
           INGA.AS         SHEL          XOM         ^AEX        ^GSPC  \
count  2063.000000  2063.000000  2063.000000  2063.000000  2063.000000   
mean     -0.000323     0.000237     0.000349    -0.000049     0.000180   
std       0.035499     0.018247     0.016456     0.014596     0.013820   
min      -0.321362    -0.115582    -0.150271    -0.095903    -0.094695   
25%      -0.013873    -0.007559    -0.006913    -0.006254    -0.004316   
50%       0.000000     0.000358     0.000000     0.000325     0.000486   
75%       0.013803     0.008898     0.007960     0.006940     0.005821   
max       0.256527     0.157214     0.158630     0.100283     0.109572   

          EURUSD=X     log_loan  
count  2063.000000  2063.000000  
mean      0.000065     0.000094  
std       0.009380     0.000076  
min      -0.143324     0.000040  
25%      -0.003573     0.000042  
50%       0.000130     0.000046  
75%       0.003868     0.000158  
max       0.159632     0.000240  


**Quantitative Assessment**

In [ ]:
print("\nFinal dataframe: log_returns")
print(log_returns.head())


Final dataframe for risk analysis: log_returns
             INGA.AS      SHEL       XOM      ^AEX     ^GSPC  EURUSD=X  \
Date                                                                     
2006-01-04  0.023097  0.010260  0.001709  0.006923  0.003666  0.006543   
2006-01-05 -0.012319 -0.012136 -0.004964 -0.004211  0.000016 -0.000412   
2006-01-06  0.006678  0.018918  0.019540  0.006702  0.009356  0.003882   
2006-01-09  0.008615 -0.001383 -0.000505  0.003960  0.003650 -0.006530   
2006-01-10 -0.025732 -0.002773  0.007714 -0.008926 -0.000357 -0.000905   

            log_loan  
Date                  
2006-01-04  0.000202  
2006-01-05  0.000202  
2006-01-06  0.000203  
2006-01-09  0.000204  
2006-01-10  0.000206  


In [15]:
# Historical VaR check against report
alpha = 0.01
n = len(log_returns)

print(f"n={n}, alpha={alpha}")
print(f"Final dataframe used: log_returns")

reported_hs_var = {
    'INGA.AS': 11.3571,
    'SHEL': 5.4736,
    'XOM': 4.6385,
    '^AEX': 4.4184,
    '^GSPC': 4.4274,
    'EURUSD=X': 1.8977,
    'log_loan': 0.0
}

print(f"\n{'Asset':<12} {'hs_var':>10} {'report':>10} {'match':>10}")
print("-" * 46)

for col in log_returns.columns:
    losses = -log_returns[col].dropna().values
    hs_var = np.percentile(losses, 99, method='linear') * 100
    if abs(hs_var) < 5e-4:
        hs_var = 0.0
    rv = reported_hs_var.get(col, np.nan)
    print(f"{col:<12} {hs_var:>10.4f} {rv:>10.4f} {str(np.isclose(hs_var, rv, atol=1e-3)):>10}")

n=2063, alpha=0.01
Final dataframe used: log_returns

Asset            hs_var     report      match
----------------------------------------------
INGA.AS         11.3571    11.3571       True
SHEL             5.4735     5.4736       True
XOM              4.6385     4.6385       True
^AEX             4.4184     4.4184       True
^GSPC            4.4274     4.4274       True
EURUSD=X         1.8977     1.8977       True
log_loan        -0.0040     0.0000      False
